### 1.1 Предварительный анализ

**Входы**

- Загруженные данные и короткие описания столбцов

**Выходы**

- Понятные описания полей и их значений
- Разделение столбцов по типам

**Что делаем**

Смотрим, что записано в каждом столбце и как читать его значения

##### Что можно понять из описаний полей

У автора датасета описания короткие: не везде указаны единицы измерения и расшифровки категорий. Посмотрим, какие значения есть в таблице, и уточним, что означает каждый столбец

##### Типы и значения столбцов

Для числовых столбцов смотрим минимум и максимум, для категориальных — список значений

In [ ]:
# столбец считаем категориальным, если он нечисловой или у него мало различных значений
MAX_CATEGORIES = 10

def is_categorical(column):
    values = data[column]
    return not pd.api.types.is_numeric_dtype(values) or values.nunique() <= MAX_CATEGORIES


def domain(column):
    values = data[column].dropna()
    if is_categorical(column):
        return '{' + ', '.join(str(v) for v in sorted(values.unique())) + '}'
    return f'[{fmt_number(values.min())}, {fmt_number(values.max())}]'


columns_info = pd.DataFrame(index=data.columns)
columns_info['Тип в данных'] = data.dtypes.astype(str)
columns_info['Домен'] = columns_info.index.map(domain)

categorical = columns_info.index.map(is_categorical)
display(columns_info[~categorical].style.set_caption('Числовые столбцы'))
display(columns_info[categorical].style.set_caption('Категориальные столбцы'))

##### Что означают столбцы

Сопоставим описания автора с тем, что увидели в данных

- **Числовые признаки** — количественные величины
- **Номинальные признаки** — категории без порядка
- **Порядковые признаки** — упорядоченные категории
- **Бинарные признаки** имеют два значения. При анализе связей будем считать их номинальными

**Дефолт** — ситуация, когда заёмщик не выполняет обязательства по кредиту: перестаёт платить или допускает существенную просрочку

<table>
<thead><tr><th>Столбец</th><th>Как понимаем значения</th><th>Тип</th></tr></thead>
<tbody>
<tr><td><code>person_age</code></td><td>возраст заёмщика, лет</td><td>Числовой</td></tr>
<tr><td><code>person_income</code></td><td>годовой доход</td><td>Числовой</td></tr>
<tr><td><code>person_home_ownership</code></td><td>жильё:<br>• RENT — снимает<br>• OWN — своё<br>• MORTGAGE — в ипотеке<br>• OTHER — другое</td><td>Номинальный</td></tr>
<tr><td><code>person_emp_length</code></td><td>стаж на текущей работе, лет</td><td>Числовой</td></tr>
<tr><td><code>loan_intent</code></td><td>цель кредита:<br>• PERSONAL — личные нужды<br>• EDUCATION — образование<br>• MEDICAL — лечение<br>• VENTURE — бизнес<br>• HOMEIMPROVEMENT — ремонт жилья<br>• DEBTCONSOLIDATION — рефинансирование других долгов</td><td>Номинальный</td></tr>
<tr><td><code>loan_grade</code></td><td>грейд займа от A (надёжный) до G (рискованный)</td><td>Порядковый</td></tr>
<tr><td><code>loan_amnt</code></td><td>сумма кредита</td><td>Числовой</td></tr>
<tr><td><code>loan_int_rate</code></td><td>процентная ставка, % годовых</td><td>Числовой</td></tr>
<tr><td><code>loan_status</code></td><td>статус кредита:<br>• 0 — платит<br>• 1 — дефолт</td><td>Бинарный</td></tr>
<tr><td><code>loan_percent_income</code></td><td>доля кредита от годового дохода</td><td>Числовой</td></tr>
<tr><td><code>cb_person_default_on_file</code></td><td>были ли дефолты раньше (по данным кредитного бюро):<br>• Y — были<br>• N — не было</td><td>Бинарный</td></tr>
<tr><td><code>cb_person_cred_hist_length</code></td><td>длина кредитной истории, лет</td><td>Числовой</td></tr>
</tbody>
</table>

In [ ]:
# порядок категорий из данных не выводится — задаём вручную
ORDINAL_COLUMNS = {'loan_grade'}

def semantic_type(column):
    if not is_categorical(column):
        return 'числовой'
    if data[column].nunique() == 2:
        return 'бинарный'
    if column in ORDINAL_COLUMNS:
        return 'порядковый'
    return 'номинальный'


columns_info['Фактический тип'] = columns_info.index.map(semantic_type)
